# 12 - Inference Pipeline Demo

**Phase 4 - Model Integration & Pipeline Development**

Unified pipeline tying the whole SmartVision AI system together:

1. Load YOLOv8 from `best.pt` and the champion classifier from its native `.keras` file
2. Run YOLOv8 detection on a test image
3. Apply NMS + confidence filtering
4. Parse each box as `[x_min, y_min, x_max, y_max]` (matching the corner format confirmed for `detection-datasets/coco` - see project report)
5. Crop with defensive padding, resize, and scale pixels for the classifier
6. Re-verify the crop with the EfficientNetB0 `.keras` model
7. Print the combined YOLO + classifier verdict for each detection

This is the same logic that powers the Streamlit app's Object Detection page.

### 1. Install dependencies and mount Drive

In [ ]:
!pip install -q datasets pyyaml tqdm scikit-learn matplotlib pandas opencv-python-headless ultralytics tensorflow pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# ---------------------------------------------------------------------------
# Project configuration - shared across every SmartVision AI notebook.
# All notebooks read/write under this same Google Drive folder so that
# work done in one notebook (e.g. dataset collection) is available to the
# next one (e.g. training), even across separate Colab sessions.
# ---------------------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/SmartVisionAI"

RAW_DATA_DIR = os.path.join(BASE_DIR, "raw_data")
RAW_IMAGES_DIR = os.path.join(RAW_DATA_DIR, "images")
RAW_ANNOTATIONS_PATH = os.path.join(RAW_DATA_DIR, "annotations.json")

CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")
CLASSIFICATION_TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
CLASSIFICATION_VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
CLASSIFICATION_TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

DETECTION_DIR = os.path.join(BASE_DIR, "detection")
DETECTION_IMAGES_DIR = os.path.join(DETECTION_DIR, "images")
DETECTION_LABELS_DIR = os.path.join(DETECTION_DIR, "labels")
DETECTION_YAML_PATH = os.path.join(DETECTION_DIR, "data.yaml")

MODELS_DIR = os.path.join(BASE_DIR, "models")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")

for d in [BASE_DIR, RAW_DATA_DIR, RAW_IMAGES_DIR, CLASSIFICATION_DIR, DETECTION_DIR, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# The 25 selected COCO classes (must match COCO category names exactly)
SELECTED_CLASSES = [
    # Vehicles (6)
    "car", "truck", "bus", "motorcycle", "bicycle", "airplane",
    # Person (1)
    "person",
    # Outdoor (3)
    "traffic light", "stop sign", "bench",
    # Animals (6)
    "dog", "cat", "horse", "bird", "cow", "elephant",
    # Kitchen & food (5)
    "bottle", "cup", "bowl", "pizza", "cake",
    # Furniture & indoor (4)
    "chair", "couch", "bed", "potted plant",
]
assert len(SELECTED_CLASSES) == 25

CLASS_TO_IDX = {name: i for i, name in enumerate(SELECTED_CLASSES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(SELECTED_CLASSES)}

def safe_name(class_name):
    return class_name.replace(" ", "_")

IMAGES_PER_CLASS = 350        # -> 8,750 images total (up from 100/class to fight overfitting)
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

CLS_IMG_SIZE = 224            # Classification input resolution (single-resolution throughout)
FINE_TUNE_IMG_SIZE = 384      # Unused by classifier training (reverted to single-resolution); kept for compatibility
YOLO_IMG_SIZE = 640
BATCH_SIZE = 32                # Stage 1 batch size
BATCH_SIZE_STAGE2 = 16         # Smaller batch at 384x384 to fit GPU memory (~2.9x pixels/image)

HF_DATASET_NAME = "detection-datasets/coco"

print("BASE_DIR:", BASE_DIR)
print("Classes:", len(SELECTED_CLASSES))


### 2. Load the YOLOv8 detector (`.pt`) and the champion classifier (`.keras`)

YOLOv8 weights stay in Ultralytics' native PyTorch `.pt` format - that's the format the framework trains, saves, and reloads with, and it's what Ultralytics' own `YOLO(path)` loader expects. The classifier uses Keras's native `.keras` format, which serializes the full model (architecture + regularizers + weights) as a single file - no `custom_objects` argument needed on reload, since every layer used in the hardened head (GlobalAveragePooling2D, Dense with L2 regularizer, BatchNormalization, Dropout) is a standard Keras layer with a built-in `get_config()`.

In [ ]:
from ultralytics import YOLO
import tensorflow as tf
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import json

# --- Step 1: load YOLOv8 from best.pt ---
YOLO_WEIGHTS_PATH = os.path.join(MODELS_DIR, "yolo_smartvision", "weights", "best.pt")
yolo_model = YOLO(YOLO_WEIGHTS_PATH)

# --- Step 1: load the champion classifier from its native .keras file ---
# Whichever model 09_Compare_Classification_Models.ipynb recommended is loaded here.
# Falls back to EfficientNetB0 by name if no summary file is present yet.
summary_path = os.path.join(OUTPUTS_DIR, "model_selection_summary.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        classifier_key = json.load(f)["recommended_model"]
else:
    classifier_key = "efficientnetb0"

CLASSIFIER_WEIGHTS_PATH = os.path.join(MODELS_DIR, f"{classifier_key}_best.keras")
classifier_model = tf.keras.models.load_model(CLASSIFIER_WEIGHTS_PATH)

PREPROCESS_FNS = {
    "vgg16": tf.keras.applications.vgg16.preprocess_input,
    "resnet50": tf.keras.applications.resnet50.preprocess_input,
    "mobilenetv2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "efficientnetb0": tf.keras.applications.efficientnet.preprocess_input,
}
classifier_preprocess = PREPROCESS_FNS[classifier_key]

# The classifier's expected input size is read straight off the loaded model, so
# this works whether it was ultimately saved at 224px or the 384px fine-tune size.
CLASSIFIER_INPUT_SIZE = classifier_model.input_shape[1]

print(f"Loaded YOLO detector from: {YOLO_WEIGHTS_PATH}")
print(f"Loaded champion classifier: {classifier_key} (input size {CLASSIFIER_INPUT_SIZE}px)")


### 3. Unified detect -> NMS -> filter -> crop -> classify pipeline

`yolo_model.predict(..., conf=..., iou=...)` applies confidence filtering and NMS internally (Steps 2-3), so the detections returned are already deduplicated and thresholded. Everything from Step 4 onward (corner-format parsing, padded crop, resize, classifier re-verification) is implemented explicitly below.

In [ ]:
CONFIDENCE_THRESHOLD = 0.5
NMS_IOU_THRESHOLD = 0.45
CROP_PADDING_RATIO = 0.08  # 8% padding on each side of the detected box


def run_smartvision_pipeline(image_path, confidence=CONFIDENCE_THRESHOLD, iou=NMS_IOU_THRESHOLD):
    """
    Full SmartVision AI inference pipeline for a single image.
    Returns (PIL.Image original, list of detection dicts).
    """
    image = Image.open(image_path).convert("RGB")
    img_w, img_h = image.size

    # --- Steps 2 & 3: YOLOv8 detection with built-in confidence filter + NMS ---
    results = yolo_model.predict(
        source=np.array(image), conf=confidence, iou=iou, verbose=False,
    )
    result = results[0]

    detections = []
    for box in result.boxes:
        # --- Step 4: parse coordinates as [x_min, y_min, x_max, y_max] ---
        # Ultralytics' box.xyxy is already in this corner format natively - it
        # matches the same [x_min, y_min, x_max, y_max] convention confirmed for
        # detection-datasets/coco's raw annotations in the project report.
        x_min, y_min, x_max, y_max = box.xyxy[0].tolist()
        yolo_cls_id = int(box.cls[0])
        yolo_confidence = float(box.conf[0])
        yolo_label = result.names.get(yolo_cls_id, str(yolo_cls_id))

        # --- Step 5a: defensive padding around the box before cropping ---
        box_w = x_max - x_min
        box_h = y_max - y_min
        pad_x = box_w * CROP_PADDING_RATIO
        pad_y = box_h * CROP_PADDING_RATIO
        crop_x1 = max(0, x_min - pad_x)
        crop_y1 = max(0, y_min - pad_y)
        crop_x2 = min(img_w, x_max + pad_x)
        crop_y2 = min(img_h, y_max + pad_y)

        # --- Step 5b: crop, resize to classifier input size, scale pixels ---
        crop = image.crop((crop_x1, crop_y1, crop_x2, crop_y2))
        crop_resized = crop.resize((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE))
        crop_array = np.expand_dims(np.array(crop_resized).astype("float32"), axis=0)
        crop_array = classifier_preprocess(crop_array)  # model-specific pixel scaling

        # --- Step 6: re-verify with the classifier ---
        class_probs = classifier_model.predict(crop_array, verbose=0)[0]
        top_idx = int(np.argmax(class_probs))
        classifier_label = SELECTED_CLASSES[top_idx]
        classifier_confidence = float(class_probs[top_idx])

        detections.append({
            "box": [x_min, y_min, x_max, y_max],
            "yolo_label": yolo_label,
            "yolo_confidence": yolo_confidence,
            "classifier_label": classifier_label,
            "classifier_confidence": classifier_confidence,
        })

    return image, detections


### 4. Draw annotated boxes with both YOLO and classifier labels

In [ ]:
def draw_detections(image, detections):
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 14)
    except Exception:
        font = ImageFont.load_default()

    for det in detections:
        x1, y1, x2, y2 = det["box"]
        label = f"{det['yolo_label']} -> {det['classifier_label']}"
        draw.rectangle([x1, y1, x2, y2], outline=(29, 158, 117), width=3)
        draw.rectangle([x1, max(0, y1 - 18), x1 + len(label) * 7, y1], fill=(29, 158, 117))
        draw.text((x1 + 3, max(0, y1 - 16)), label, fill=(255, 255, 255), font=font)
    return annotated


### 5. Run it on a sample test image

**Step 7:** prints the final combined output for every detection: `Detected [YOLO Label] with X% confidence -> Re-verified by {classifier} as [Classifier Label] with Y% confidence`.

In [ ]:
import glob
import matplotlib.pyplot as plt

sample_images = glob.glob(os.path.join(DETECTION_IMAGES_DIR, "test", "*.jpg"))
assert sample_images, "Run 10_YOLO_Dataset_Preparation.ipynb first."

image, detections = run_smartvision_pipeline(sample_images[0])
annotated = draw_detections(image, detections)

plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"{len(detections)} object(s) detected")
plt.show()

# --- Step 7: final combined print output ---
classifier_display_name = classifier_key.replace("efficientnetb0", "EfficientNetB0") \
                                          .replace("resnet50", "ResNet50") \
                                          .replace("vgg16", "VGG16") \
                                          .replace("mobilenetv2", "MobileNetV2")

if not detections:
    print("No objects detected above the confidence threshold.")
for det in detections:
    print(
        f"Detected {det['yolo_label']} with {det['yolo_confidence']*100:.1f}% confidence "
        f"-> Re-verified by {classifier_display_name} as {det['classifier_label']} "
        f"with {det['classifier_confidence']*100:.1f}% confidence"
    )


### Next step: the Streamlit app

This same detect -> NMS -> filter -> pad -> crop -> classify logic is what powers `app/inference_pipeline.py` in the Streamlit application. After downloading `SmartVisionAI/models/` and `SmartVisionAI/outputs/` from Google Drive into the local project's `models/` and `outputs/` folders, run:

```bash
streamlit run app/Home.py
```
See the project README for full deployment instructions to Hugging Face Spaces.